# 00 — Setup check

**What this tests:** that your environment is ready — configuration loads from `ai/.env`,
the LLM layer works (fake **and** real Gemini), local embeddings load, Tavily is reachable,
and the disk cache is active.

Run top to bottom with `LLM_PROVIDER=fake` (no network, no key) and later with
`LLM_PROVIDER=gemini` + a real `GEMINI_API_KEY`. Cells that need a key skip themselves
with an explanation when it is missing.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Make oppradar_ai importable whether the kernel starts in ai/ or ai/notebooks/.
AI_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(AI_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_ROOT))

import pandas as pd
from IPython.display import display

from oppradar_ai.config import get_settings, setup_logging
from oppradar_ai.llm import get_llm, get_stats, reset_stats

setup_logging("INFO")
settings = get_settings().ensure_dirs()

pd.Series(settings.model_dump(mode="json"), name="value").rename_axis("setting").to_frame()

## Step 1 — LLM layer with the fake provider (always works, zero network)

The fake provider returns deterministic, schema-valid canned data. It proves the whole
pipeline (schema validation, caching, stats) works before you spend a single free-tier token.

In [ ]:
from oppradar_ai.schemas import Opportunity

fake = get_llm("fake")
reset_stats()

opp = fake.structured("Extract: Machine Learning Intern at Ooredoo, apply by 2026-11-30.", Opportunity)
print(opp.model_dump_json(indent=2))

print("\ntext ->", fake.text("What is OppRadar?"))

stats = get_stats()
print(f"\nprovider={fake.name} model={fake.model} | calls={stats.calls} cached={stats.cached} errors={stats.errors}")

## Step 2 — Real Gemini Flash (needs `GEMINI_API_KEY` in `ai/.env`)

Set `LLM_PROVIDER=gemini` (or just fill the key). If no key is configured this cell
skips itself — nothing to fix when you want to stay offline.

In [ ]:
from oppradar_ai.llm import LLMError

if settings.gemini_key_value:
    try:
        gemini = get_llm("gemini")
        result = gemini.structured(
            "Name one real, well-known scholarship programme for African students.",
            Opportunity,
        )
        print("GEMINI OK ->", result.title, "|", result.organizer)
    except LLMError as exc:
        print("Gemini call failed (check key / model name):", exc)
else:
    print("No GEMINI_API_KEY configured — set it in ai/.env to test the real LLM.")

## Step 3 — Local embeddings (sentence-transformers)

Default model is `BAAI/bge-m3` (~2.3 GB, first run downloads it). If that is too heavy,
set `EMBEDDING_MODEL=sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`
(~470 MB) in `ai/.env`. A healthy result is a similarity **above ~0.4** for the two
related sentences below.

In [ ]:
import time

import numpy as np

try:
    from sentence_transformers import SentenceTransformer

    t0 = time.time()
    model = SentenceTransformer(settings.embedding_model)
    load_s = time.time() - t0

    t0 = time.time()
    vectors = model.encode([
        "remote machine learning internship in Algiers",
        "offre de stage deep learning a Alger",
    ])
    embed_s = time.time() - t0
    sim = float(vectors[0] @ vectors[1] / (np.linalg.norm(vectors[0]) * np.linalg.norm(vectors[1])))

    print(f"model={settings.embedding_model}")
    print(f"loaded in {load_s:.1f}s, embedded 2 sentences in {embed_s:.2f}s, dim={vectors.shape[1]}")
    print(f"cosine similarity = {sim:.3f}   (expected > 0.4 for two related sentences)")
except Exception as exc:
    print("Embeddings unavailable:", type(exc).__name__, "-", exc)

## Step 4 — Tavily web discovery (needs `TAVILY_API_KEY`)

Only used by ingestion to *discover* links. Every result is cached on disk, so a free
tier goes a long way. Skipped when no key is set.

In [ ]:
if settings.tavily_key_value:
    from tavily import TavilyClient

    client = TavilyClient(api_key=settings.tavily_key_value)
    response = client.search("scholarships for African students 2027", max_results=3)
    for row in response.get("results", []):
        print("-", row.get("title"), "|", row.get("url"))
else:
    print("No TAVILY_API_KEY configured — link discovery will be skipped (see notebook 03).")

## Step 5 — Cache stats and session summary

The disk cache stores every LLM call keyed by `(provider, model, prompt)`. Re-running a
cell with the same prompt should **not** increase `calls` — only `cached`.

In [ ]:
cache_dir = settings.cache_dir / "llm"
cache_files = list(cache_dir.glob("*.json")) if cache_dir.exists() else []
cache_mb = sum(f.stat().st_size for f in cache_files) / 1e6
stats = get_stats()

summary = pd.DataFrame([
    {"component": "Config", "detail": f"provider={settings.llm_provider}, model={settings.gemini_model}"},
    {"component": "FakeLLM", "detail": "structured + text OK offline"},
    {"component": "LLM disk cache", "detail": f"{len(cache_files)} entries, {cache_mb:.2f} MB in {cache_dir}"},
    {"component": "This session", "detail": f"calls={stats.calls}, cached={stats.cached}, errors={stats.errors}"},
])
display(summary)

## Summary — what to check

| Check | Expected |
| --- | --- |
| Settings table | `llm_provider` is `fake` (or `gemini` once configured); no secret values visible |
| Step 1 | A valid `Opportunity` JSON, and running the cell twice keeps `calls=1, cached=1` |
| Step 2 | Either `GEMINI OK -> ...` or a clean skip message |
| Step 3 | Similarity **> 0.4**; on the very first run a large model download happens |
| Step 4 | Three Tavily results, or a clean skip message |
| Step 5 | Cache entry count grows when new prompts are used, not when old ones repeat |

If all rows pass, continue with `01_document_intake.ipynb`.